In [ ]:
import numpy
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

In [ ]:
df = pd.read_csv('qoute_dataset.csv')

In [ ]:
df.head()

In [ ]:
df.shape

In [ ]:
quotes = df['quote']
quotes.str.lower()

In [ ]:
import string
translator = str.maketrans('', '', string.punctuation)
quotes = quotes.apply(lambda x: x.translate(translator))

In [ ]:
import nltk
from nltk.stem import WordNetLemmatizer, PorterStemmer
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

# Download necessary NLTK data (if not already downloaded)
nltk.download('punkt', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('stopwords', quiet=True)
nltk.download('punkt_tab', quiet=True)


### Tokenization

First, let's tokenize the `quotes` series. Tokenization breaks down text into individual words or tokens.

In [ ]:
tokenized_quotes = quotes.apply(word_tokenize)

print("Tokenized Quotes (first 5):")
display(tokenized_quotes.head())


### Lemmatization

Next, we'll apply lemmatization to the tokenized quotes. Lemmatization reduces words to their base or dictionary form (lemma), which is often a valid word.

In [ ]:
# Initialize the lemmatizer and define stopwords
lemmatizer = WordNetLemmatizer()
stop_words = set(stopwords.words('english'))

def lemmatize_and_filter_text(tokens):
    lemmatized_words = [lemmatizer.lemmatize(word) for word in tokens if word.lower() not in stop_words]
    return ' '.join(lemmatized_words)

# Apply lemmatization to the tokenized quotes
lemmatized_quotes_processed = tokenized_quotes.apply(lemmatize_and_filter_text)

print("Lemmatized Quotes (first 5):")
display(lemmatized_quotes_processed.head())


### Stemming

Finally, we'll apply stemming. Stemming reduces words to their root form (stem), which may not always be a valid word, but is useful for reducing inflected words to a common base form.

In [ ]:
# Initialize the Porter Stemmer
stemmer = PorterStemmer()

def stem_and_filter_text(tokens):
    stemmed_words = [stemmer.stem(word) for word in tokens if word.lower() not in stop_words]
    return ' '.join(stemmed_words)

# Apply stemming to the tokenized quotes
stemmed_quotes_processed = tokenized_quotes.apply(stem_and_filter_text)

print("Stemmed Quotes (first 5):")
display(stemmed_quotes_processed.head())


In [ ]:
!pip install wordcloud

In [ ]:
from wordcloud import WordCloud
import matplotlib.pyplot as plt

# Combine all lemmatized quotes into a single string
all_lemmatized_words = ' '.join(lemmatized_quotes_processed)

# Generate the word cloud
wordcloud = WordCloud(width = 800, height = 800,
                background_color ='white',
                stopwords = set(stopwords.words('english')),
                min_font_size = 10).generate(all_lemmatized_words)

# Display the generated image:
plt.figure(figsize = (8, 8), facecolor = None)
plt.imshow(wordcloud)
plt.axis("off")
plt.tight_layout(pad = 0)

plt.show()


### Word Cloud with Custom Shape

To create a word cloud in a specific shape, we can use a mask image. The white areas of the mask image will determine the shape of the word cloud, while black areas will be transparent.

**Instructions for creating a mask image:**
1. Find or create a simple black and white image of the desired shape (e.g., a cloud).
2. Ensure the background of the shape is white and the foreground (the shape itself) is black. If your image has a white shape on a black background, you might need to invert the colors.
3. Save the image as a PNG file (e.g., `cloud_mask.png`) and upload it to your Colab environment. For example, a simple cloud silhouette with a white background will work best.

In [ ]:
import numpy as np
from PIL import Image
from wordcloud import WordCloud, STOPWORDS
import matplotlib.pyplot as plt

# Load the mask image
try:
    cloud_mask = np.array(Image.open("cloud_mask.png"))
except FileNotFoundError:
    print("Error: 'cloud_mask.png' not found. Please upload a cloud-shaped PNG image to your Colab environment.")
    print("You can download one or create your own. Ensure the cloud shape is white and the background is black.")
    # Fallback to default if mask is not found to prevent error
    cloud_mask = None

# Combine all lemmatized quotes into a single string
all_lemmatized_words = ' '.join(lemmatized_quotes_processed)

# Define a list of stopwords to exclude common words
stopwords = set(STOPWORDS)
# You might want to add custom stopwords if needed
# stopwords.update(['said', 'like', 'one', 'good'])

# Generate the word cloud with the mask
wordcloud_shaped = WordCloud(width=800, height=800,
                        background_color='white',
                        stopwords=stopwords,
                        mask=cloud_mask,
                        contour_width=3,
                        contour_color='steelblue',
                        min_font_size=10).generate(all_lemmatized_words)

# Display the generated image:
plt.figure(figsize=(10, 10), facecolor=None)
plt.imshow(wordcloud_shaped, interpolation='bilinear')
plt.axis("off")
plt.tight_layout(pad=0)
plt.show()

### Preparing Data for LSTM

For an LSTM model, we need to convert our processed text into numerical sequences. We'll create a tokenizer to map words to integers and then form sequences of a fixed length for training. Each sequence will predict the next word.

In [ ]:
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras.preprocessing.sequence import pad_sequences

# Combine all lemmatized quotes into a single string to create sequences
corpus = ' '.join(lemmatized_quotes_processed)

# Tokenize the entire corpus
tokenizer = Tokenizer()
tokenizer.fit_on_texts([corpus])

# Get the vocabulary size
total_words = len(tokenizer.word_index) + 1

# Create input sequences
input_sequences = []
for line in lemmatized_quotes_processed:
    token_list = tokenizer.texts_to_sequences([line])[0]
    for i in range(1, len(token_list)):
        n_gram_sequence = token_list[:i+1]
        input_sequences.append(n_gram_sequence)

# Pad sequences
# Find the maximum sequence length
max_sequence_len = max([len(x) for x in input_sequences])

# Pad all sequences to the same length
padded_sequences = np.array(pad_sequences(input_sequences, maxlen=max_sequence_len, padding='pre'))

# Create predictors (X) and labels (y)
X, y = padded_sequences[:,:-1], padded_sequences[:,-1]
y = tf.keras.utils.to_categorical(y, num_classes=total_words)

print(f"Total words in vocabulary: {total_words}")
print(f"Max sequence length: {max_sequence_len}")
print(f"Shape of X (input sequences): {X.shape}")
print(f"Shape of y (target words): {y.shape}")

### Define and Compile the LSTM Model

Now, we'll define a simple LSTM model using Keras. It will consist of an Embedding layer, an LSTM layer, and a Dense output layer with softmax activation to predict the next word.

In [ ]:
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Embedding, LSTM, Dense

# Define the model
model = Sequential()
model.add(Embedding(total_words, 100, input_length=max_sequence_len-1))
model.add(LSTM(150))
model.add(Dense(total_words, activation='softmax'))

# Compile the model
model.compile(loss='categorical_crossentropy', optimizer='adam', metrics=['accuracy'])

model.summary()

### Train the LSTM Model

Finally, we will train the LSTM model using the prepared data, with 50 epochs and a batch size of 16 as requested.

In [ ]:
epochs = 50
batch_size = 16

history = model.fit(X, y, epochs=epochs, batch_size=batch_size, verbose=1)

print("\nModel training complete.")

Epoch 1/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 49s 17ms/step - accuracy: 0.0107 - loss: 7.8740
Epoch 2/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 44s 18ms/step - accuracy: 0.0128 - loss: 7.4774
Epoch 3/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 81s 17ms/step - accuracy: 0.0175 - loss: 7.2001
Epoch 4/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - accuracy: 0.0287 - loss: 6.7693
Epoch 5/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 43s 17ms/step - accuracy: 0.0484 - loss: 6.2010
Epoch 6/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 42s 17ms/step - accuracy: 0.0897 - loss: 5.5631
Epoch 7/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 43s 17ms/step - accuracy: 0.1563 - loss: 4.9221
Epoch 8/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 82s 17ms/step - accuracy: 0.2396 - loss: 4.3102
Epoch 9/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 43s 17ms/step - accuracy: 0.3224 - loss: 3.7579
Epoch 10/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 43s 17ms/step - accuracy: 0.3998 - loss: 3.2737
Epoch 11/50
2491/2491 ━━━━━━━━━━━━━━━━━━━━ 44s 18ms/step - accuracy: 0.4664 - loss: 2.8512
Epoch 12

In [ ]:
model.save('lstm_model.h5')

In [ ]:
from tensorflow.keras.models import load_model
import numpy as np

# Load the trained model
model = load_model('lstm_model.h5')

def generate_text(model, tokenizer, max_sequence_len, seed_text, next_words):
    for _ in range(next_words):
        token_list = tokenizer.texts_to_sequences([seed_text])[0]
        token_list = pad_sequences([token_list], maxlen=max_sequence_len-1, padding='pre')
        predicted = np.argmax(model.predict(token_list, verbose=0), axis=-1)

        output_word = ""
        for word, index in tokenizer.word_index.items():
            if index == predicted:
                output_word = word
                break
        seed_text += " " + output_word
    return seed_text



Seed: world created process
Generated: world created process thinking changed without changing thinking


In [ ]:
# Example usage:
seed_text = "Any fool can know. The point"
next_words = 5
generated_sentence = generate_text(model, tokenizer, max_sequence_len, seed_text, next_words)
print(f"Seed: {seed_text}")
print(f"Generated: {generated_sentence}")

Seed: Any fool can know. The point
Generated: Any fool can know. The point man love people walk take
